# Payments vs the TM channel view: inflow and outflow by rail (v1.1)

**PNC Treasury Management · DSI · Payment Knowledge Graph (PKG)**

**Question.** For TM customers, how much of the money the TM channel view (`clhhd01p_clh.cdm_tm_cor_trans_agg_v`)
shows moving through ACH, wire, RTP and checks do we see in the PKG payments, in and out, month by month? And
which pieces are missing?

**How.** Payments are summed per customer (power id) and month from the edge tables (`SOURCE = "edges"`, the same
money as `neo4j_payments` after the 5C/6C duplicates and debit card are removed) or straight from `neo4j_payments`
(`SOURCE = "payments"`, slower). Each payment is split by rail (category number), direction and who is on the other
side:

| Other side | Meaning |
|---|---|
| ext | outside PNC (a counterparty) |
| int | another PNC customer (a different power id) |
| same | the same customer: own accounts, same mdm id or another mdm id under the same power id |

The channel view is one row per account and month with dollars and units per channel. Its accounts are summed per
power id. Earlier, the deposit reconciliation estimated the PKG side from rail shares; here both sides are summed
from the raw amounts, and the view's columns are tested piece by piece (for example ACH credits received vs ACH
debits originated) to find exactly which kinds of payment PKG holds.

| § | Question |
|---|---|
| 0 | Inputs, window, the view's columns (dollars, units, anything that looks like transfers between customers) |
| 1 | Payments and channel view per customer and month (cached work file) |
| 2 | Who is in which source, by month |
| 3 | Month alignment: does the view's month match the payment month? |
| 4 | Totals by rail and direction, overall and by month |
| 5 | Which pieces match: every combination of view columns against every slice of payments |
| 6 | Counts: payment counts against the view's units |
| 7 | Between customers: internal and same-customer payments, and view columns that may hold them |
| 8 | Where the gap sits: customer size and the largest gaps |
| 9 | Set aside PNC's own accounts and the large processors; the 30 largest gaps over all rails |
| 10 | Timing of originated ACH: matches over 1, 2 and 3 months; first and last posting days |
| 11 | Checks paid: missing, partial or matched; by check size |
| 12 | Coverage per customer and month, written as a table |
| 13 | Answer sheet |

Read-only on the sources; writes a work file under `WORK_DIR`, csv files and figures under `OUT_DIR`, and the
coverage table `COVERAGE_TABLE` (§12).

**v1.1 (2026-11-16):** v1.0 found the 80% is not a missing kind of payment: ordinary customers match almost exactly,
and a few accounts (PNC's own, payroll and check processors, FMR, Walker & Dunlop) hold most of the gap. Adds §9-§12.
A rerun with the same settings reuses the work file from v1.0, so only the new sections take time.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os, re, sys, time, math, warnings, threading, contextlib
import numpy as np
import pandas as pd

for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
               ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import DateType, TimestampType, NumericType

DB = "bdahd01p_dlcdi1_cdi_tm"


def _pv(label, digits, scopes):
    return (label, digits, scopes)


_EI, _EIS = ["ext", "int"], ["ext", "int", "same"]
CONFIG = dict(
    SOURCE      = "edges",                     # "edges" (pkg_edge_monthly, fast) or "payments" (neo4j_payments, slow)
    EDGE_TABLE  = f"{DB}.pkg_edge_monthly",
    PAY_TABLE   = "dsihd01p_dsi.neo4j_payments",
    DEDUP_TABLE = f"{DB}.pkg_dedup_5c6c",      # 6C twins of 5C payments, removed in payments mode
    DIM_TABLE   = f"{DB}.pkg_cust_dim",        # mdm_id -> customer_pwr_id (TM customers have one)
    CHAN_TABLE  = "clhhd01p_clh.cdm_tm_cor_trans_agg_v",
    CHAN_ID     = "cust_pwr_id", CHAN_KEY = "time_key", CHAN_ACCT = "acct_dim_key",
    FIRST_MONTH = "2024-01", LAST_MONTH = None,   # None = last month in both sources
    # view columns tested per rail and direction ("core" = the columns used for the headline totals) and the payment
    # slices they are tested against: (label, category numbers, other sides)
    RAILS = {
        "ACH in":    dict(core=["ach_credits_recd_amt", "ach_debits_orig_amt"],
                          view=["ach_credits_recd_amt", "ach_debits_orig_amt", "elctr_pymt_credit_amt", "elctr_check_orig_amt"],
                          pkg=[_pv("originated outside PNC (3), ext", ["3"], ["ext"]),
                               _pv("originated through PNC (1, 2), ext", ["1", "2"], ["ext"]),
                               _pv("originated through PNC (1, 2), ext + other customers", ["1", "2"], _EI),
                               _pv("all ACH, ext", ["1", "2", "3"], ["ext"]),
                               _pv("all ACH, ext + other customers", ["1", "2", "3"], _EI),
                               _pv("all ACH, everything", ["1", "2", "3"], _EIS)]),
        "ACH out":   dict(core=["ach_credits_orig_amt", "ach_debits_recd_amt"],
                          view=["ach_credits_orig_amt", "ach_debits_recd_amt", "elctr_pymt_debit_amt", "elctr_check_rcvd_amt"],
                          pkg=[_pv("originated outside PNC (3), ext", ["3"], ["ext"]),
                               _pv("originated through PNC (1, 2), ext", ["1", "2"], ["ext"]),
                               _pv("originated through PNC (1, 2), ext + other customers", ["1", "2"], _EI),
                               _pv("all ACH, ext", ["1", "2", "3"], ["ext"]),
                               _pv("all ACH, ext + other customers", ["1", "2", "3"], _EI),
                               _pv("all ACH, everything", ["1", "2", "3"], _EIS)]),
        "wire in":   dict(core=["dom_wire_incoming_amt", "intl_wire_incoming_amt"],
                          view=["dom_wire_incoming_amt", "intl_wire_incoming_amt"],
                          pkg=[_pv("wire, ext", ["4"], ["ext"]), _pv("wire, ext + other customers", ["4"], _EI),
                               _pv("wire, everything", ["4"], _EIS)]),
        "wire out":  dict(core=["dom_wire_outgoing_amt", "intl_wire_outgoing_amt"],
                          view=["dom_wire_outgoing_amt", "intl_wire_outgoing_amt"],
                          pkg=[_pv("wire, ext", ["4"], ["ext"]), _pv("wire, ext + other customers", ["4"], _EI),
                               _pv("wire, everything", ["4"], _EIS)]),
        "RTP in":    dict(core=["realtime_pymt_credit_amt"], view=["realtime_pymt_credit_amt"],
                          pkg=[_pv("RTP_PRT (5), ext", ["5"], ["ext"]), _pv("RTP_P2P (6), ext", ["6"], ["ext"]),
                               _pv("all RTP, ext", ["5", "6"], ["ext"]), _pv("all RTP, ext + other customers", ["5", "6"], _EI),
                               _pv("all RTP, everything", ["5", "6"], _EIS)]),
        "RTP out":   dict(core=["realtime_pymt_debit_amt"], view=["realtime_pymt_debit_amt"],
                          pkg=[_pv("RTP_PRT (5), ext", ["5"], ["ext"]), _pv("RTP_P2P (6), ext", ["6"], ["ext"]),
                               _pv("all RTP, ext", ["5", "6"], ["ext"]), _pv("all RTP, ext + other customers", ["5", "6"], _EI),
                               _pv("all RTP, everything", ["5", "6"], _EIS)]),
        "check in":  dict(core=["check_dep_lockbox_amt", "check_dep_remote_amt"],
                          view=["check_dep_lockbox_amt", "check_dep_remote_amt", "branch_dep_amt", "elctr_check_orig_amt"],
                          pkg=[_pv("check, ext", ["7"], ["ext"]), _pv("check, ext + other customers", ["7"], _EI),
                               _pv("check, everything", ["7"], _EIS)]),
        "check out": dict(core=["check_disbursement_amt"], view=["check_disbursement_amt", "elctr_check_rcvd_amt"],
                          pkg=[_pv("check, ext", ["7"], ["ext"]), _pv("check, ext + other customers", ["7"], _EI),
                               _pv("check, everything", ["7"], _EIS)]),
    },
    RAIL_OF = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "wire", "5": "RTP", "6": "RTP", "7": "check", "8": "card", "9": "PCARD"},
    INTERNAL_REGEX = r"book|xfer|trnsf|trsf|transfer|intra|internal|on_?us|zba|sweep|conc|between|dda_to|acct_to",
    UNIT_SUFFIXES = ["_units", "_unit", "_cnt", "_count", "_qty", "_nbr", "_num", "_vol", "_items", "_itm", "_no"],
    TOL = (0.01, 0.05),        # a customer-month matches when the two sides differ by 1% (5%) or less
    TOP_COMBOS = 4,            # combinations shown per rail and direction in §5
    TOP_GAPS = 5,              # customers shown per rail and direction in §8
    # §9: name groups set aside (upper-case customer name, regex, first match wins)
    GROUPS = {
        "PNC own": r"\bPNC\b",
        "processors and servicers": r"PAYLOCITY|PAYCOM|TELECHECK|FISERV|STRIPE|PAYPAL|\bADP\b|PAYCHEX|CERIDIAN|SITUS|WORLDPAY|"
                                    r"GLOBAL PAYMENTS|FIRST DATA|BILL\.COM|INTUIT|EMPOWER",
        "other named large accounts": r"^FMR LLC|WALKER & DUNLOP",
    },
    # §10: pairs tested over 1, 2 and 3 months: (label, view columns, category numbers, other sides, direction)
    TIMING_PAIRS = [
        ("ACH in: credits received + debits originated vs all ACH", ["ach_credits_recd_amt", "ach_debits_orig_amt"], ["1", "2", "3"], _EIS, "in"),
        ("ACH in: credits received vs ACH originated outside PNC (3)", ["ach_credits_recd_amt"], ["3"], _EIS, "in"),
        ("ACH in: debits originated vs ACH through PNC (1, 2)", ["ach_debits_orig_amt"], ["1", "2"], _EIS, "in"),
        ("ACH out: credits originated + debits received vs all ACH", ["ach_credits_orig_amt", "ach_debits_recd_amt"], ["1", "2", "3"], _EIS, "out"),
        ("ACH out: debits received vs ACH originated outside PNC (3)", ["ach_debits_recd_amt"], ["3"], _EIS, "out"),
        ("ACH out: credits originated vs ACH through PNC (1, 2)", ["ach_credits_orig_amt"], ["1", "2"], _EIS, "out"),
        ("wire out vs wire", ["dom_wire_outgoing_amt", "intl_wire_outgoing_amt"], ["4"], _EIS, "out"),
        ("checks paid vs checks", ["check_disbursement_amt"], ["7"], _EIS, "out"),
    ],
    EDGE_DAILY = f"{DB}.pkg_edge_daily",
    TIMING_DAYS_MONTHS = 6,    # §10 posting-day check on the last N months (0 = skip)
    # §12
    WRITE_COVERAGE = True,
    COVERAGE_TABLE = f"{DB}.pkg_cust_rail_coverage",
    ID = "customer_pwr_id",
    WORK_DIR = "/user/pk36814/pkg_work/rail_recon",
    REBUILD = False,           # True = rebuild the work file even when one with the same settings exists
    OUT_DIR = "../eda/rail_recon",
    SHUFFLE_PARTITIONS = 400,
)
ID = CONFIG["ID"]

spark = (SparkSession.builder.appName("pkg_rail_recon")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.yarn.am.memory", "4g")
         .config("spark.yarn.am.memoryOverhead", "1g")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.kryoserializer.buffer.max", "128m")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
               "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
               "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False

def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def __init__(self):
        self.lines, self.tables, self.h = [], [], None

    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80))

    def _draw(self):
        if self.h is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()
TIMINGS = []


@contextlib.contextmanager
def step(name):
    """Timed step with a live line: elapsed minutes and Spark tasks of the running stage."""
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        TIMINGS.append((name, round(mins, 2)))
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


GATES = []


def gate(name, ok, detail, action=""):
    GATES.append(dict(gate=name, result="PASS" if ok else "FAIL", detail=detail, action=action))


def gates_frame():
    return pd.DataFrame(GATES, columns=["gate", "result", "detail", "action"])



OUT_DIR = CONFIG["OUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)
FIG_INDEX, ANSWERS = [], []
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": "#b8b7b0", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110,
})


def savefig(fig, name, caption):
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX.append(dict(file=f"{name}.png", caption=caption))


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))


def table_months(tbl, col="month"):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().pipe(_nodec).iloc[:, 0]
        m = sorted({x.split(f"{col}=")[1].split("/")[0] for x in p if f"{col}=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(str(r[0]) for r in spark.table(tbl).select(col).distinct().collect())


def mi_col(c):
    c = F.col(c) if isinstance(c, str) else c
    return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int")).cast("int")


def mi2m(i):
    i = int(i)
    return f"{(i - 1) // 12:04d}-{(i - 1) % 12 + 1:02d}"


def m2mi(m):
    return int(m[:4]) * 12 + int(m[5:7])


def _nodec(pdf):
    """Spark DECIMAL columns arrive as Python Decimal objects; turn them into floats."""
    import decimal
    for c in pdf.columns:
        if pdf[c].dtype == object and pdf[c].map(lambda v: isinstance(v, decimal.Decimal)).any():
            pdf[c] = pdf[c].map(lambda v: float(v) if isinstance(v, decimal.Decimal) else v).astype(float)
    return pdf


def pct_tbl(df, cols):
    df = df.copy()
    for c in cols:
        df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{100 * v:.1f}%")
    return df


BANDS = [(-np.inf, 1_000, "under 1K"), (1_000, 10_000, "1K-10K"), (10_000, 100_000, "10K-100K"),
         (100_000, 1_000_000, "100K-1M"), (1_000_000, np.inf, "1M and over")]


def band(col):
    e = F.lit(BANDS[-1][2])
    for lo, hi, lab in reversed(BANDS[:-1]):
        e = F.when(F.col(col) < hi, F.lit(lab)).otherwise(e)
    return e


import hashlib, json, itertools


def pid_col(c, dtype):
    """power id as text without leading zeros; blank, '-1' and '0' = missing"""
    x = F.col(c)
    if dtype not in ("string",) and not dtype.startswith("varchar") and not dtype.startswith("char"):
        x = x.cast("decimal(38,0)")
    s = F.regexp_replace(F.trim(x.cast("string")), r"^0+", "")
    return F.when(s.isNull() | (s == "") | (s == "-1"), F.lit(None)).otherwise(s)


def months_between(a, b):
    return [mi2m(i) for i in range(m2mi(a), m2mi(b) + 1)]


print("setup done")

## §0 · Inputs, window and the view's columns

The window is every month both sources hold from `FIRST_MONTH`. The view's period key is turned into a month; if a
month has many keys (daily rows) they are summed, if it has one or a few (month-end snapshots) the last is kept.
Each dollar column is paired with its units column where one exists (same name with `_units`, `_cnt` and the like).

In [ ]:
CELL.start("§0 · Inputs and window")
with step("schemas, months and period keys"):
    dim = spark.table(CONFIG["DIM_TABLE"])
    chv = spark.table(CONFIG["CHAN_TABLE"])
    CHT = dict(chv.dtypes)
    for c in [CONFIG["CHAN_ID"], CONFIG["CHAN_KEY"]]:
        if c not in CHT:
            raise ValueError(f"{CONFIG['CHAN_TABLE']} has no column {c}")
    if CONFIG["SOURCE"] == "edges":
        src_months = table_months(CONFIG["EDGE_TABLE"])
    else:
        try:
            _pm = spark.sql(f"SHOW PARTITIONS {CONFIG['PAY_TABLE']}").toPandas().iloc[:, 0]
            src_months = sorted({re.search(r"trans_dt=(\d{4}-\d{2})", x).group(1) for x in _pm if re.search(r"trans_dt=(\d{4}-\d{2})", x)})
        except Exception:
            src_months = []
        if not src_months:
            src_months = sorted(r[0] for r in spark.table(CONFIG["PAY_TABLE"]).select(F.substring("trans_dt", 1, 7)).distinct().collect())
    K = CONFIG["CHAN_KEY"]
    _kt = CHT[K]
    _kexpr = F.date_format(F.col(K), "yyyyMMdd") if _kt in ("date", "timestamp") else F.regexp_replace(F.col(K).cast("string"), "-", "")
    _keys = (chv.select(_kexpr.alias("key")).groupBy("key").agg(F.count(F.lit(1)).alias("rows")).toPandas())
    _keys["key"] = _keys.key.astype(str).str.replace(r"\.0$", "", regex=True)
    _keys["month"] = np.where(_keys.key.str.len().isin([6, 8]), _keys.key.str[:4] + "-" + _keys.key.str[4:6], None)
    _keys = _keys[_keys.month.notna()]
    _kpm = _keys.groupby("month").agg(keys=("key", "nunique"), rows=("rows", "sum"), last_key=("key", "max")).reset_index()
    DAILY_KEYS = bool(len(_kpm) and _kpm["keys"].median() >= 15)
    view_months = sorted(_kpm.month)
    last = CONFIG["LAST_MONTH"] or min(max(src_months), max(view_months))
    MONTHS = [m for m in months_between(CONFIG["FIRST_MONTH"], last) if m in src_months and m in view_months]
    if not MONTHS:
        raise ValueError("No month is in both the payments and the channel view.")
    USE_KEYS = (_keys[_keys.month.isin(MONTHS)].key.tolist() if DAILY_KEYS
                else _kpm[_kpm.month.isin(MONTHS)].last_key.tolist())
    # dollar and unit columns
    _num = [c for c, t in chv.dtypes if re.match(r"(int|bigint|smallint|tinyint|double|float|decimal)", t)
            and c not in (CONFIG["CHAN_ID"], K, CONFIG["CHAN_ACCT"])]
    AMT_COLS = [c for c in _num if c.endswith("_amt")]
    UNIT_OF = {}
    for c in AMT_COLS:
        base = c[: -len("_amt")]
        for s in CONFIG["UNIT_SUFFIXES"]:
            if base + s in _num:
                UNIT_OF[c] = base + s
                break
    INT_COLS = [c for c in _num if re.search(CONFIG["INTERNAL_REGEX"], c, re.I)]
    _cfg_missing = sorted({c for r in CONFIG["RAILS"].values() for c in r["view"] if c not in AMT_COLS})
    _other_num = [c for c in _num if c not in AMT_COLS and c not in UNIT_OF.values()]
_win = pd.DataFrame([
    dict(item="payments source", value=f"{CONFIG['SOURCE']}: {CONFIG['EDGE_TABLE'] if CONFIG['SOURCE'] == 'edges' else CONFIG['PAY_TABLE']}"),
    dict(item="window", value=f"{MONTHS[0]} .. {MONTHS[-1]} ({len(MONTHS)} months)"),
    dict(item="view period key", value=f"{K} ({_kt}); {'daily rows, summed per month' if DAILY_KEYS else 'snapshots, last key of each month'}; "
                                       f"median keys per month {int(_kpm['keys'].median()) if len(_kpm) else 0}"),
    dict(item="view months", value=f"{view_months[0]} .. {view_months[-1]} ({len(view_months)})"),
    dict(item="view dollar columns", value=f"{len(AMT_COLS)}, {len(UNIT_OF)} with a units column"),
    dict(item="configured view columns not found", value=", ".join(_cfg_missing) or "none"),
    dict(item="columns that may hold transfers between customers or accounts", value=", ".join(INT_COLS) or "none found"),
    dict(item="other numeric columns", value=", ".join(_other_num) or "none"),
])
_cols = pd.DataFrame([dict(dollar_column=c, units_column=UNIT_OF.get(c, ""),
                           tested_in=", ".join(k for k, r in CONFIG["RAILS"].items() if c in r["view"]))
                      for c in AMT_COLS])
_cols.to_csv(os.path.join(OUT_DIR, "view_columns.csv"), index=False)
CELL.show(("Window and inputs", _win), ("View dollar columns, their units column and where they are tested", _cols),
          ("Period keys, last 6 months of the window", _kpm[_kpm.month.isin(MONTHS)].tail(6)))
ans("Window", f"{MONTHS[0]} .. {MONTHS[-1]} ({len(MONTHS)} months); payments from {CONFIG['SOURCE']}")
ans("View columns that may hold transfers between customers", ", ".join(INT_COLS) or "none found by name")

## §1 · Payments and channel view per customer and month

**Payments.** Every payment is read from each PKG customer's side: money out for the payer, money in for the
receiver (a payment between two PNC customers counts for both). The mdm id is turned into its power id through
`pkg_cust_dim`; customers without a power id (not TM) are left out. Rails come from the category number (1-3 ACH,
4 wire, 5 RTP_PRT, 6 RTP_P2P, 7 check, 9 PCARD); debit card is not in the edges.

**View.** Every account row of the window is summed per power id and month (accounts with a missing power id are
counted in §2 but not matched).

Both are joined into one row per power id and month and cached under `WORK_DIR`.

In [ ]:
CELL.start("§1 · Payments and view per customer and month")
SIG = hashlib.md5(json.dumps({**{k: CONFIG[k] for k in ["SOURCE", "EDGE_TABLE", "PAY_TABLE", "DEDUP_TABLE", "DIM_TABLE", "CHAN_TABLE",
                                                     "CHAN_ID", "CHAN_KEY"]}, **{"months": MONTHS}}, default=str).encode()).hexdigest()[:8]


def cached(sdf_fn, name):
    path = f"{CONFIG['WORK_DIR']}/sig={SIG}/{name}"
    if not CONFIG["REBUILD"]:
        try:
            out = spark.read.parquet(path)
            out.limit(1).count()
            CELL.log(f"reused work file: {name}")
            return out
        except Exception:
            pass
    sdf_fn().write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


_dt = dict(dim.dtypes)
MAP = (dim.select(F.col("mdm_id").cast("string").alias("mdm"), pid_col(ID, _dt[ID]).alias("pid"))
       .where(F.col("pid").isNotNull()).dropDuplicates(["mdm"]))
NAME = (dim.select(pid_col(ID, _dt[ID]).alias("pid"), F.col("customer_name")).where(F.col("pid").isNotNull())
        .groupBy("pid").agg(F.min("customer_name").alias("customer_name"))) if "customer_name" in dim.columns else None


def _perspective():
    if CONFIG["SOURCE"] == "edges":
        e = (spark.table(CONFIG["EDGE_TABLE"]).where(F.col("month").isin(MONTHS))
             .select("month", "src_id", "src_kind", "dst_id", "dst_kind", "category", F.col("is_self").cast("boolean").alias("is_self"),
                     F.col("amount").cast("double").alias("amt"), F.col("n_txn").cast("long").alias("n")))
    else:
        p = (spark.table(CONFIG["PAY_TABLE"])
             .where((F.col("trans_dt") >= f"{MONTHS[0]}-01") & (F.col("trans_dt") <= f"{MONTHS[-1]}-31"))
             .where(~F.col("category").startswith("8")))
        try:
            dd = spark.table(CONFIG["DEDUP_TABLE"]).select(F.col("trans_id_drop").cast("string").alias("trans_id")).distinct()
            p = p.join(dd, p["trans_id"].cast("string") == dd["trans_id"], "left_anti")
        except Exception:
            CELL.log("dedup table not readable: 6C twins are kept")
        _cu = lambda c: F.when(F.trim(F.col(c).cast("string")) != "", F.col(c).cast("string"))
        e = p.select(F.substring("trans_dt", 1, 7).alias("month"),
                     F.coalesce(_cu("mdm_id_pays"), F.lit("CPTY")).alias("src_id"),
                     F.when(_cu("mdm_id_pays").isNotNull(), "CUST").otherwise("CPTY").alias("src_kind"),
                     F.coalesce(_cu("mdm_id_receives"), F.lit("CPTY")).alias("dst_id"),
                     F.when(_cu("mdm_id_receives").isNotNull(), "CUST").otherwise("CPTY").alias("dst_kind"),
                     "category", (F.coalesce(_cu("mdm_id_pays"), F.lit("a")) == F.coalesce(_cu("mdm_id_receives"), F.lit("b"))).alias("is_self"),
                     F.abs(F.col("trans_amt").cast("double")).alias("amt"), F.lit(1).cast("long").alias("n"))
    code = F.regexp_extract(F.col("category").cast("string"), r"^\s*(\d[A-Ca-c]\d?)", 1)
    out = e.where(F.col("src_kind") == "CUST").select("month", F.col("src_id").alias("mdm"), F.col("dst_kind").alias("okind"),
                                                      F.col("dst_id").alias("other"), code.alias("code"), F.lit("out").alias("dir"),
                                                      "is_self", "amt", "n")
    inn = e.where(F.col("dst_kind") == "CUST").select("month", F.col("dst_id").alias("mdm"), F.col("src_kind").alias("okind"),
                                                      F.col("src_id").alias("other"), code.alias("code"), F.lit("in").alias("dir"),
                                                      "is_self", "amt", "n")
    r = out.unionByName(inn).withColumn("mdm", F.col("mdm").cast("string")).withColumn("other", F.col("other").cast("string"))
    r = r.join(MAP, "mdm", "inner")
    om = MAP.select(F.col("mdm").alias("other"), F.col("pid").alias("opid"))
    r = r.join(om, "other", "left")
    scope = (F.when(F.col("okind") != "CUST", "ext")
             .when(F.coalesce(F.col("is_self"), F.lit(False)) | (F.col("opid") == F.col("pid")), "same").otherwise("int"))
    return (r.withColumn("scope", scope)
            .withColumn("k", F.concat(F.lit("k"), F.upper("code"), F.lit("_"), F.col("dir"), F.lit("_"), F.col("scope")))
            .groupBy("pid", "month", "k").agg(F.sum("amt").alias("a"), F.sum("n").alias("n")))


def _view():
    v = chv.where(_kexpr.isin(USE_KEYS))
    _m = F.concat(F.substring(_kexpr, 1, 4), F.lit("-"), F.substring(_kexpr, 5, 2))
    cols = AMT_COLS + list(UNIT_OF.values())
    return (v.select(pid_col(CONFIG["CHAN_ID"], CHT[CONFIG["CHAN_ID"]]).alias("pid"), _m.alias("month"),
                     F.col(CONFIG["CHAN_ACCT"]).alias("acct") if CONFIG["CHAN_ACCT"] in CHT else F.lit(1).alias("acct"),
                     *[F.coalesce(F.col(c).cast("double"), F.lit(0.0)).alias(c) for c in cols])
            .groupBy("pid", "month").agg(F.countDistinct("acct").alias("n_acct"), *[F.sum(c).alias(c) for c in cols]))


with step("payments per customer, month, category, direction and other side"):
    LONG = cached(_perspective, "pkg_long")
    KEYS = sorted(r[0] for r in LONG.select("k").distinct().collect())
with step("channel view per customer and month"):
    VIEW = cached(_view, "view")
with step("one row per customer and month"):
    def _join():
        wide = LONG.groupBy("pid", "month").pivot("k", KEYS).agg(F.sum("a").alias("a"), F.sum("n").alias("n"))
        wide = wide.withColumn("in_pkg", F.lit(1))
        j = (VIEW.where(F.col("pid").isNotNull()).withColumn("in_view", F.lit(1))
             .join(wide, ["pid", "month"], "full"))
        fill = [c for c in j.columns if c not in ("pid", "month")]
        return j.na.fill(0, fill)
    J = cached(_join, "joined").persist()
    NROW = J.count()
KA = [f"{k}_a" for k in KEYS]


def pkg_sum(digits, scopes, d, kind="a"):
    ks = [f"{k}_{kind}" for k in KEYS
          if k[1] in digits and k.split("_")[1] == d and k.split("_")[2] in scopes]
    return sum((F.col(c) for c in ks), F.lit(0.0)) if ks else F.lit(0.0)


def view_sum(cols, kind="a"):
    cols = [c for c in cols if c in AMT_COLS] if kind == "a" else [UNIT_OF[c] for c in cols if c in UNIT_OF]
    return sum((F.col(c) for c in cols), F.lit(0.0)) if cols else F.lit(0.0)


RAILS = {k: dict(r, view=[c for c in r["view"] if c in AMT_COLS], core=[c for c in r["core"] if c in AMT_COLS])
         for k, r in CONFIG["RAILS"].items()}
RAILS = {k: r for k, r in RAILS.items() if r["view"]}
DIG = {k: sorted({d for p in r["pkg"] for d in p[1]}) for k, r in RAILS.items()}
DIR = {k: k.split()[-1] for k in RAILS}
_cat = (LONG.groupBy("k").agg(F.sum("a").alias("dollars"), F.sum("n").alias("payments"), F.countDistinct("pid").alias("customers"))
        .toPandas().pipe(_nodec))
_cat["category"] = _cat.k.str[1:].str.split("_").str[0]
_cat["direction"] = _cat.k.str.split("_").str[1]
_cat["other_side"] = _cat.k.str.split("_").str[2]
_cat["rail"] = _cat.category.str[0].map(CONFIG["RAIL_OF"])
_cat = _cat.sort_values(["rail", "category", "direction", "other_side"])[["rail", "category", "direction", "other_side", "customers", "payments", "dollars"]]
_cat.to_csv(os.path.join(OUT_DIR, "pkg_by_category.csv"), index=False)
CELL.show(("Rows", pd.DataFrame([dict(customer_months=NROW, payment_slices=len(KEYS), work_file=f"{CONFIG['WORK_DIR']}/sig={SIG}")])),
          ("TM customers' payments by category, direction and other side (whole window)", _cat))

## §2 · Who is in which source

Per month: power ids in both sources, only in the view, only in payments, and the dollars each group carries.
Accounts in the view with no power id are counted separately.

In [ ]:
CELL.start("§2 · Who is in which source")
with step("population by month"):
    _all_in = sum((F.col(f"{k}_a") for k in KEYS if k.split("_")[1] == "in" and k[1] != "9"), F.lit(0.0))
    _all_out = sum((F.col(f"{k}_a") for k in KEYS if k.split("_")[1] == "out" and k[1] != "9"), F.lit(0.0))
    _vin = view_sum(sorted({c for k, r in RAILS.items() if DIR[k] == "in" for c in r["core"]}))
    _vout = view_sum(sorted({c for k, r in RAILS.items() if DIR[k] == "out" for c in r["core"]}))
    POP = (J.withColumn("grp", F.when((F.col("in_view") == 1) & (F.col("in_pkg") == 1), "both")
                        .when(F.col("in_view") == 1, "view only").otherwise("payments only"))
           .groupBy("month", "grp").agg(F.count(F.lit(1)).alias("customers"), F.sum(_vin).alias("view_in"), F.sum(_vout).alias("view_out"),
                                        F.sum(_all_in).alias("pkg_in"), F.sum(_all_out).alias("pkg_out"))
           .toPandas().pipe(_nodec))
    _nopid = (chv.where(_kexpr.isin(USE_KEYS)).where(pid_col(CONFIG["CHAN_ID"], CHT[CONFIG["CHAN_ID"]]).isNull())
              .agg(F.count(F.lit(1)).alias("rows"), F.sum(view_sum(sorted({c for r in RAILS.values() for c in r["core"]}))).alias("dollars"))
              .toPandas().pipe(_nodec))
_pp = POP.pivot_table(index="month", columns="grp", values="customers", aggfunc="sum").fillna(0).reset_index()
_tot = POP.groupby("grp")[["customers", "view_in", "view_out", "pkg_in", "pkg_out"]].sum().reset_index()
_tot["share_of_view_in"] = _tot.view_in / max(_tot.view_in.sum(), 1)
_tot["share_of_pkg_in"] = _tot.pkg_in / max(_tot.pkg_in.sum(), 1)
POP.to_csv(os.path.join(OUT_DIR, "population_by_month.csv"), index=False)
CELL.show(("Customer-months by source (whole window); view dollars = core columns of the four rails", pct_tbl(_tot, ["share_of_view_in", "share_of_pkg_in"])),
          ("Customers per month", _pp.tail(12)),
          ("View rows with no power id (left out)", _nopid))
_b = _tot.set_index("grp")
ans("Customer-months in both / view only / payments only",
    " / ".join(f"{int(_b.customers.get(g, 0)):,}" for g in ["both", "view only", "payments only"]))
ans("Share of view inflow (core columns) at customers with no PKG payments", f"{_b.view_in.get('view only', 0) / max(_b.view_in.sum(), 1):.1%}")

## §3 · Month alignment

The view could report a month's activity under the next month's key (some CDM facts do). Customer-months in both
sources are compared with the view shifted by one month each way; the share of matches (within 5%) should peak at
shift 0 if the months line up.

In [ ]:
CELL.start("§3 · Month alignment")
with step("matches at shifts -1, 0, +1"):
    BOTH = J.where((F.col("in_view") == 1) & (F.col("in_pkg") == 1))
    _rows = []
    _vcols = sorted({c for r in RAILS.values() for c in r["core"]})
    _V = J.where(F.col("in_view") == 1).select("pid", "month", *_vcols)
    _P = J.where(F.col("in_pkg") == 1).select("pid", "month", *KA)
    for s in (-1, 0, 1):
        _Vs = _V.withColumn("month", F.concat_ws("-", ((mi_col("month") - s - 1) / 12).cast("int").cast("string"),
                                                  F.lpad((((mi_col("month") - s - 1) % 12) + 1).cast("string"), 2, "0")))
        jj = _Vs.join(_P, ["pid", "month"], "inner")
        aggs = []
        for k, r in RAILS.items():
            v, p = view_sum(r["core"]), pkg_sum(DIG[k], ["ext", "int"], DIR[k])
            ok = (v > 0) | (p > 0)
            aggs += [F.sum(F.when(ok, 1).otherwise(0)).alias(f"{k}|n"),
                     F.sum(F.when(ok & (F.abs(p - v) <= 0.05 * F.greatest(p, v)), 1).otherwise(0)).alias(f"{k}|m")]
        rr = jj.agg(*aggs).toPandas().iloc[0]
        for k in RAILS:
            _rows.append(dict(rail=k, shift=s, customer_months=int(rr[f"{k}|n"] or 0),
                              match_5pct=(rr[f"{k}|m"] or 0) / max(rr[f"{k}|n"] or 0, 1)))
ALIGN = pd.DataFrame(_rows)
_al = ALIGN.pivot_table(index="rail", columns="shift", values="match_5pct").reset_index()
_al.columns = ["rail"] + [f"shift {c:+d}" for c in _al.columns[1:]]
_best_shift = ALIGN.groupby("shift").match_5pct.mean().idxmax()
CELL.show(("Share of customer-months matching within 5% (view core columns vs payments ext + other customers)",
           pct_tbl(_al, [c for c in _al.columns if c.startswith("shift")])))
gate("view months line up with payment months", _best_shift == 0, f"best shift {_best_shift:+d}",
     "if not 0, the view reports activity under another month's key: shift it before reading §4-§8")
ans("Month alignment (best shift of the view)", f"{_best_shift:+d}")

## §4 · Totals by rail and direction

Customer-months in both sources. View = the core columns (ACH: credits received + debits originated in, credits
originated + debits received out; wires: domestic + international; RTP; checks: lockbox + remote in, check
disbursements out). Payments split by other side. PCARD has no counterpart in the view and is shown for scale.
Saved: f01 (share by month).

In [ ]:
CELL.start("§4 · Totals by rail and direction")
with step("totals and monthly shares"):
    aggs = []
    for k, r in RAILS.items():
        aggs += [F.sum(view_sum(r["core"])).alias(f"{k}|view")] + \
                [F.sum(pkg_sum(DIG[k], [s], DIR[k])).alias(f"{k}|{s}") for s in ("ext", "int", "same")]
    for d in ("in", "out"):
        aggs.append(F.sum(pkg_sum(["9"], ["ext", "int", "same"], d)).alias(f"PCARD {d}|ext"))
    MON = BOTH.groupBy("month").agg(*aggs).orderBy("month").toPandas().pipe(_nodec)
_tot = MON.drop(columns="month").sum()
TOT = []
for k in RAILS:
    v, e, i, s = (_tot[f"{k}|{x}"] for x in ("view", "ext", "int", "same"))
    TOT.append(dict(rail=k, view=v, pkg_ext=e, pkg_other_customers=i, pkg_same_customer=s,
                    ext_share_of_view=e / v if v else np.nan, ext_int_share_of_view=(e + i) / v if v else np.nan,
                    all_share_of_view=(e + i + s) / v if v else np.nan))
TOT = pd.DataFrame(TOT)
_pc = pd.DataFrame([dict(rail=f"PCARD {d}", pkg_dollars=_tot.get(f"PCARD {d}|ext", 0.0)) for d in ("in", "out")])
SHM = MON[["month"]].copy()
for k in RAILS:
    SHM[k] = (MON[f"{k}|ext"] + MON[f"{k}|int"]) / MON[f"{k}|view"].replace(0, np.nan)
TOT.to_csv(os.path.join(OUT_DIR, "totals_by_rail.csv"), index=False)
SHM.to_csv(os.path.join(OUT_DIR, "share_by_month.csv"), index=False)
fig, ax = plt.subplots(figsize=(9, 4.2))
_cl = [C1, C2, C3, C4, "#7c5cd6", "#d64d8a", "#5c8a2a", "#8a6a2a"]
for c, k in zip(_cl, RAILS):
    ax.plot(pd.to_datetime(SHM.month), SHM[k], color=c, lw=2, marker="o", ms=3, label=k)
ax.axhline(1, color=INK2, lw=0.8, ls="--")
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Payments (ext + other customers) as a share of the channel view, by month")
ax.legend(ncol=4, fontsize=8, loc="lower left")
savefig(fig, "f01_share_by_month", "Payments as a share of the channel view by rail and month (customers in both sources)")
CELL.show(("Whole window, customers in both sources (dollars)",
           pct_tbl(TOT, ["ext_share_of_view", "ext_int_share_of_view", "all_share_of_view"])),
          ("By month: payments (ext + other customers) as a share of the view", pct_tbl(SHM.copy(), list(RAILS))),
          ("PCARD (no counterpart in the view)", _pc))
_pf = lambda v: "n/a" if pd.isna(v) else f"{v:.1%}"
for _, r in TOT.iterrows():
    ans(f"{r.rail}: payments as a share of the view (ext / ext + other customers / everything)",
        f"{_pf(r.ext_share_of_view)} / {_pf(r.ext_int_share_of_view)} / {_pf(r.all_share_of_view)}")

## §5 · Which pieces match

For each rail and direction, every combination of the view's columns is compared with every slice of payments,
customer-month by customer-month (customers in both sources, months where either side is above zero):

- **match 1% / 5%**: share of customer-months where the two differ by 1% (5%) or less. An exact definition match
  shows as a high share at 1%; a share near zero means the two hold different things even if the totals look close.
- **payments ÷ view**: dollar ratio over the window.

The best combination per rail (highest match at 1%) says what PKG holds. Example: if ACH in matches "credits
received" against "originated outside PNC (3)", PKG has ACH credits but not the debits the customer originates.

In [ ]:
CELL.start("§5 · Which pieces match")
COMBO = []
with step("every view combination against every payment slice"):
    for k, r in RAILS.items():
        combos = [list(c) for n in range(1, len(r["view"]) + 1) for c in itertools.combinations(r["view"], n)]
        aggs, idx = [], []
        for ci, cols in enumerate(combos):
            v = view_sum(cols)
            for pi, (lab, digs, scopes) in enumerate(r["pkg"]):
                p = pkg_sum(digs, scopes, DIR[k])
                ok = (v > 0) | (p > 0)
                t = f"c{ci}p{pi}"
                aggs += [F.sum(F.when(ok, 1).otherwise(0)).alias(f"{t}n"),
                         F.sum(F.when(ok & (F.abs(p - v) <= CONFIG["TOL"][0] * F.greatest(p, v)), 1).otherwise(0)).alias(f"{t}m1"),
                         F.sum(F.when(ok & (F.abs(p - v) <= CONFIG["TOL"][1] * F.greatest(p, v)), 1).otherwise(0)).alias(f"{t}m5"),
                         F.sum(F.when(ok & (p < v * (1 - CONFIG["TOL"][1])), 1).otherwise(0)).alias(f"{t}lo"),
                         F.sum(v).alias(f"{t}v"), F.sum(p).alias(f"{t}p")]
                idx.append((t, cols, lab))
        rr = BOTH.agg(*aggs).toPandas().pipe(_nodec).iloc[0]
        for t, cols, lab in idx:
            n = rr[f"{t}n"] or 0
            COMBO.append(dict(rail=k, view_columns=" + ".join(c.replace("_amt", "") for c in cols), payments=lab,
                              customer_months=int(n), match_1pct=(rr[f"{t}m1"] or 0) / max(n, 1),
                              match_5pct=(rr[f"{t}m5"] or 0) / max(n, 1), payments_lower=(rr[f"{t}lo"] or 0) / max(n, 1),
                              payments_over_view=(rr[f"{t}p"] or 0) / (rr[f"{t}v"] or np.nan), view_dollars=rr[f"{t}v"] or 0.0,
                              _cols=cols, _single=len(cols) == 1))
COMBO = pd.DataFrame(COMBO)
COMBO["_m1"] = COMBO.match_1pct.round(2)
COMBO["_m5"] = COMBO.match_5pct.round(2)
COMBO = COMBO.sort_values(["rail", "_m1", "_m5", "view_dollars"], ascending=[True, False, False, False])
_hid = ["_cols", "_single", "_m1", "_m5"]
COMBO.drop(columns=_hid).to_csv(os.path.join(OUT_DIR, "combinations.csv"), index=False)
COMBO = COMBO[COMBO.customer_months > 0]
BEST = COMBO.groupby("rail", sort=False).head(1).set_index("rail").reindex([k for k in RAILS if k in set(COMBO.rail)]).reset_index()
SINGLE = (COMBO[COMBO._single].groupby(["rail", "view_columns"], sort=False).head(1)
          .assign(_o=lambda d: d.rail.map({k: i for i, k in enumerate(RAILS)})).sort_values(["_o", "view_dollars"], ascending=[True, False])
          .drop(columns=_hid + ["_o"]))
_top = COMBO.groupby("rail", sort=False).head(CONFIG["TOP_COMBOS"]).drop(columns=_hid)
_top = _top.assign(_o=_top.rail.map({k: i for i, k in enumerate(RAILS)})).sort_values("_o", kind="stable").drop(columns="_o")
_pc5 = ["match_1pct", "match_5pct", "payments_lower", "payments_over_view"]
CELL.show(("Best match per rail and direction (highest match at 1%, then most view dollars)", pct_tbl(BEST.drop(columns=_hid), _pc5)),
          ("Each view column on its own: the payment slice it matches best", pct_tbl(SINGLE, _pc5)),
          (f"Top {CONFIG['TOP_COMBOS']} per rail and direction (all in combinations.csv)",
           pct_tbl(_top, _pc5)))
for _, r in BEST.iterrows():
    ans(f"{r.rail}: best match", f"view {r.view_columns} = payments {r.payments}: {r.match_1pct:.0%} within 1%, "
                                 f"{r.match_5pct:.0%} within 5%, payments ÷ view {r.payments_over_view:.0%}")

## §6 · Counts

For the best combination of each rail: payment count against the view's units (when every column in the combination
has a units column). Equal counts on most customer-months confirm the match; counts that agree while dollars do not
point to amounts (fees, returns); dollars that agree while counts do not point to batching (one view unit per file).

In [ ]:
CELL.start("§6 · Counts")
CNT = []
with step("counts for the best combinations"):
    for _, b in BEST.iterrows():
        k, cols = b.rail, b._cols
        if not cols or not all(c in UNIT_OF for c in cols):
            CNT.append(dict(rail=k, view_columns=b.view_columns, status="no units column for every view column"))
            continue
        lab, digs, scopes = next(p for p in RAILS[k]["pkg"] if p[0] == b.payments)
        v, p = view_sum(cols, "n"), pkg_sum(digs, scopes, DIR[k], "n")
        ok = (v > 0) | (p > 0)
        rr = BOTH.agg(F.sum(F.when(ok, 1).otherwise(0)).alias("n"), F.sum(F.when(ok & (p == v), 1).otherwise(0)).alias("eq"),
                      F.sum(F.when(ok & (F.abs(p - v) <= 0.05 * F.greatest(p, v)), 1).otherwise(0)).alias("m5"),
                      F.sum(v).alias("v"), F.sum(p).alias("p")).toPandas().pipe(_nodec).iloc[0]
        _n = rr["n"] or 0
        CNT.append(dict(rail=k, view_columns=b.view_columns, payments=b.payments, status="compared", customer_months=int(_n),
                        counts_equal=(rr["eq"] or 0) / max(_n, 1), counts_within_5pct=(rr["m5"] or 0) / max(_n, 1),
                        view_units=rr["v"], payment_count=rr["p"], count_ratio=(rr["p"] or 0) / (rr["v"] or np.nan)))
CNT = pd.DataFrame(CNT)
for c in ["counts_equal", "counts_within_5pct", "count_ratio"]:
    if c not in CNT:
        CNT[c] = np.nan
CNT.to_csv(os.path.join(OUT_DIR, "counts.csv"), index=False)
CELL.show(("Payment count against the view's units (best combination of each rail)", pct_tbl(CNT.copy(), ["counts_equal", "counts_within_5pct", "count_ratio"])))

## §7 · Between customers

Payments between two PNC customers (int) and within one customer (same: own accounts, same mdm id or another mdm id
under the same power id), by rail. The view's columns that look like transfers by name are compared with them.
Whether adding internal payments improves the match is in §5 (slices "ext + other customers" and "everything").

In [ ]:
CELL.start("§7 · Between customers")
with step("internal and same-customer payments by rail; view columns that look like transfers"):
    _ra = sorted({CONFIG["RAIL_OF"].get(k[1], "other") for k in KEYS})
    aggs = []
    for rail in _ra:
        digs = [d for d, r in CONFIG["RAIL_OF"].items() if r == rail]
        for d in ("in", "out"):
            for s in ("ext", "int", "same"):
                aggs.append(F.sum(pkg_sum(digs, [s], d)).alias(f"{rail}|{d}|{s}"))
    aggs += [F.sum(F.col(c)).alias(f"v|{c}") for c in INT_COLS]
    rr = BOTH.agg(*aggs).toPandas().pipe(_nodec).iloc[0]
    INTT = []
    for rail in _ra:
        for d in ("in", "out"):
            e, i, s = (rr[f"{rail}|{d}|{x}"] or 0.0 for x in ("ext", "int", "same"))
            if e + i + s:
                INTT.append(dict(rail=rail, direction=d, ext=e, other_customers=i, same_customer=s,
                                 other_customers_share=i / (e + i + s), same_customer_share=s / (e + i + s)))
    INTT = pd.DataFrame(INTT)
    ICMP = []
    for c in INT_COLS:
        v = F.col(c)
        for lab, d, scopes in [("all rails, other customers, in", "in", ["int"]), ("all rails, other customers, out", "out", ["int"]),
                               ("all rails, same customer, in", "in", ["same"]), ("all rails, same customer, out", "out", ["same"]),
                               ("all rails, other + same, in", "in", ["int", "same"]), ("all rails, other + same, out", "out", ["int", "same"])]:
            p = pkg_sum([x for x in CONFIG["RAIL_OF"] if x not in ("8", "9")], scopes, d)
            ok = (v > 0) | (p > 0)
            q = BOTH.agg(F.sum(F.when(ok, 1).otherwise(0)).alias("n"),
                         F.sum(F.when(ok & (F.abs(p - v) <= 0.05 * F.greatest(p, v)), 1).otherwise(0)).alias("m5"),
                         F.sum(v).alias("v"), F.sum(p).alias("p")).toPandas().pipe(_nodec).iloc[0]
            ICMP.append(dict(view_column=c, payments=lab, customer_months=int(q["n"] or 0), match_5pct=(q["m5"] or 0) / max(q["n"] or 0, 1),
                             view_dollars=q["v"], payment_dollars=q["p"]))
    ICMP = pd.DataFrame(ICMP, columns=["view_column", "payments", "customer_months", "match_5pct", "view_dollars", "payment_dollars"])
    if len(ICMP):
        ICMP = ICMP.sort_values(["view_column", "match_5pct"], ascending=[True, False])
INTT.to_csv(os.path.join(OUT_DIR, "between_customers.csv"), index=False)
CELL.show(("Payments by other side (customers in both sources, whole window)", pct_tbl(INTT.copy(), ["other_customers_share", "same_customer_share"])),
          ("View columns that look like transfers, against internal payments" if INT_COLS else
           "No view column looks like a transfer between customers or accounts by its name",
           pct_tbl(ICMP.copy(), ["match_5pct"]) if len(ICMP) else pd.DataFrame(dict(view_column=["none found"]))))
if len(INTT):
    _x = INTT.groupby("direction")[["ext", "other_customers", "same_customer"]].sum()
    ans("Payments between PNC customers (other customers / same customer) as a share of all, in",
        f"{_x.loc['in', 'other_customers'] / _x.loc['in'].sum():.1%} / {_x.loc['in', 'same_customer'] / _x.loc['in'].sum():.1%}" if "in" in _x.index else "")

## §8 · Where the gap sits

For the core view columns against payments (ext + other customers): the share by customer size (the view's monthly
dollars on that rail) and the customers with the largest gaps over the window, for spot checks.

In [ ]:
CELL.start("§8 · Where the gap sits")
with step("share by size band and largest gaps"):
    _b = []
    _g = []
    for k, r in RAILS.items():
        v, p = view_sum(r["core"]), pkg_sum(DIG[k], ["ext", "int"], DIR[k])
        x = BOTH.select("pid", "month", v.alias("v"), p.alias("p")).where((F.col("v") > 0) | (F.col("p") > 0))
        _b.append(x.withColumn("band", band("v")).groupBy("band").agg(F.count(F.lit(1)).alias("customer_months"),
                                                                      F.sum("v").alias("view"), F.sum("p").alias("pkg"))
                  .withColumn("rail", F.lit(k)).toPandas().pipe(_nodec))
        _g.append(x.groupBy("pid").agg(F.sum("v").alias("view"), F.sum("p").alias("pkg"), F.count(F.lit(1)).alias("months"))
                  .withColumn("gap", F.col("view") - F.col("pkg")).orderBy(F.desc("gap")).limit(CONFIG["TOP_GAPS"])
                  .withColumn("rail", F.lit(k)).toPandas().pipe(_nodec))
    BAND = pd.concat(_b, ignore_index=True)
    BAND["share"] = BAND.pkg / BAND.view.replace(0, np.nan)
    _order = [b[2] for b in BANDS]
    BANDP = BAND.pivot_table(index="band", columns="rail", values="share").reindex(_order).reset_index()
    BANDP = BANDP[["band"] + [k for k in RAILS if k in BANDP.columns]]
    GAPS = pd.concat(_g, ignore_index=True)
    if NAME is not None and len(GAPS):
        _nm = NAME.where(F.col("pid").isin(GAPS.pid.unique().tolist())).toPandas()
        GAPS = GAPS.merge(_nm, on="pid", how="left")
    GAPS["share"] = GAPS.pkg / GAPS.view.replace(0, np.nan)
    GAPS = GAPS[["rail", "pid"] + (["customer_name"] if "customer_name" in GAPS else []) + ["months", "view", "pkg", "gap", "share"]]
BAND.to_csv(os.path.join(OUT_DIR, "share_by_size.csv"), index=False)
GAPS.to_csv(os.path.join(OUT_DIR, "largest_gaps.csv"), index=False)
CELL.show(("Payments (ext + other customers) as a share of the view, by the customer's monthly view dollars on that rail",
           pct_tbl(BANDP.copy(), [c for c in BANDP.columns if c != "band"])),
          (f"Largest gaps over the window ({CONFIG['TOP_GAPS']} per rail; power id = customer_pwr_id without leading zeros)",
           pct_tbl(GAPS.copy(), ["share"])))

## §9 · Set aside PNC's own accounts and the large processors

§8 showed a few accounts holding most of the gap. Customers are grouped by name (`GROUPS`, first match wins) and
the share is recomputed without each group. Then the 30 customers with the largest gap over all rails together
(view core columns minus payments, everything) with their share on each rail, to sort by hand.

In [ ]:
CELL.start("§9 · Set aside PNC's own accounts and processors")


def _grp_expr():
    e = F.lit("other")
    for g, rx in reversed(list(CONFIG["GROUPS"].items())):
        e = F.when(F.upper(F.coalesce(F.col("customer_name"), F.lit(""))).rlike(rx), F.lit(g)).otherwise(e)
    return e


with step("shares by group and the largest gaps over all rails"):
    BG = (BOTH.join(NAME, "pid", "left") if NAME is not None
          else BOTH.withColumn("customer_name", F.lit(None).cast("string"))).withColumn("grp", _grp_expr())
    _ev = {k: (view_sum(r["core"]), pkg_sum(DIG[k], ["ext", "int", "same"], DIR[k])) for k, r in RAILS.items()}
    GSUM = (BG.groupBy("grp").agg(F.countDistinct("pid").alias("customers"),
                                  *[F.sum(v).alias(f"{k}|v") for k, (v, p) in _ev.items()],
                                  *[F.sum(p).alias(f"{k}|p") for k, (v, p) in _ev.items()])
            .toPandas().pipe(_nodec).set_index("grp"))
    _gs = list(CONFIG["GROUPS"])
    SETA = []
    for k in RAILS:
        v, p = GSUM[f"{k}|v"], GSUM[f"{k}|p"]
        row = dict(rail=k, view=v.sum(), all_customers=p.sum() / v.sum() if v.sum() else np.nan)
        kept = list(GSUM.index)
        for g in _gs:
            if g in kept:
                kept.remove(g)
            vv, pp = v.reindex(kept).sum(), p.reindex(kept).sum()
            row[f"without {', '.join(_gs[:_gs.index(g) + 1])}"] = pp / vv if vv else np.nan
        gap = (v - p).clip(lower=0)
        for g in _gs:
            row[f"gap held by {g}"] = gap.get(g, 0.0) / gap.sum() if gap.sum() else np.nan
        SETA.append(row)
    SETA = pd.DataFrame(SETA)
    _cust = (BG.groupBy("pid").agg(F.first("customer_name", ignorenulls=True).alias("customer_name"), F.first("grp").alias("group"),
                                   *[F.sum(v).alias(f"v{i}") for i, (k, (v, p)) in enumerate(_ev.items())],
                                   *[F.sum(p).alias(f"p{i}") for i, (k, (v, p)) in enumerate(_ev.items())]))
    _gap = sum((F.greatest(F.col(f"v{i}") - F.col(f"p{i}"), F.lit(0.0)) for i in range(len(_ev))), F.lit(0.0))
    _cust = _cust.withColumn("total_gap", _gap)
    _allgap = _cust.agg(F.sum("total_gap")).first()[0] or 0.0
    TOP30 = _cust.orderBy(F.desc("total_gap")).limit(30).toPandas().pipe(_nodec)
    TOP30["share_of_all_gaps"] = TOP30.total_gap / _allgap if _allgap else np.nan
    for i, k in enumerate(RAILS):
        TOP30[k] = TOP30[f"p{i}"] / TOP30[f"v{i}"].replace(0, np.nan)
    TOP30 = TOP30[["pid", "customer_name", "group", "total_gap", "share_of_all_gaps"] + list(RAILS)]
SETA.to_csv(os.path.join(OUT_DIR, "share_without_groups.csv"), index=False)
TOP30.to_csv(os.path.join(OUT_DIR, "top30_gaps.csv"), index=False)
_cs = GSUM[["customers"]].reset_index()
CELL.show(("Customers in each group (customers in both sources)", _cs),
          ("Payments (everything) as a share of the view core columns, setting groups aside one after another",
           pct_tbl(SETA, [c for c in SETA.columns if c not in ("rail", "view")])),
          ("30 largest gaps over all rails (rail columns = payments ÷ view for that customer; blank = no view dollars)",
           pct_tbl(TOP30, ["share_of_all_gaps"] + list(RAILS))))
ans("30 largest-gap customers: share of all dollar gaps", f"{TOP30.total_gap.sum() / _allgap:.1%}" if _allgap else "n/a")
for _, r in SETA.iterrows():
    ans(f"{r.rail}: share of the view, all customers / without all named groups",
        f"{_pf(r.all_customers)} / {_pf(r[[c for c in SETA.columns if c.startswith('without')][-1]])}")

## §10 · Timing of originated ACH

Received ACH matches customer-month by customer-month; originated ACH (debits the customer originates to collect,
credits it originates to pay) does not, though the dollar totals are close. If the view dates originated entries by
settlement day and PKG by posting day, entries near month end fall into different months, and the match should
improve when months are summed in pairs or quarters. If instead payments are missing, summing can only lower the
match (a miss in any month spoils the period). Only customers in both sources for every month of the period.

Second check from the daily edges (last `TIMING_DAYS_MONTHS` months): the share of each category's dollars posted on
the last two and first two posting days of the month. A spread-out category puts roughly 10% on each.

In [ ]:
CELL.start("§10 · Timing of originated ACH")
with step("matches summed over 1, 2 and 3 months"):
    _pairs = [(lab, [c for c in vc if c in AMT_COLS], digs, scopes, d) for lab, vc, digs, scopes, d in CONFIG["TIMING_PAIRS"]]
    _pairs = [x for x in _pairs if x[1]]
    _need = sorted({c for _, vc, _, _, _ in _pairs for c in vc})
    TIM = []
    for per_lab, n_m in [("1 month", 1), ("2 months", 2), ("quarter", 3)]:
        P = (BOTH.withColumn("per", ((mi_col("month") - 1) / n_m).cast("int"))
             .groupBy("pid", "per").agg(F.count(F.lit(1)).alias("nm"), *[F.sum(c).alias(c) for c in _need + KA])
             .where(F.col("nm") == n_m))
        aggs = []
        for i, (lab, vc, digs, scopes, d) in enumerate(_pairs):
            v, p = view_sum(vc), pkg_sum(digs, scopes, d)
            ok = (v > 0) | (p > 0)
            aggs += [F.sum(F.when(ok, 1).otherwise(0)).alias(f"n{i}"),
                     F.sum(F.when(ok & (F.abs(p - v) <= 0.01 * F.greatest(p, v)), 1).otherwise(0)).alias(f"a{i}"),
                     F.sum(F.when(ok & (F.abs(p - v) <= 0.05 * F.greatest(p, v)), 1).otherwise(0)).alias(f"b{i}")]
        rr = P.agg(*aggs).toPandas().pipe(_nodec).iloc[0]
        for i, (lab, vc, digs, scopes, d) in enumerate(_pairs):
            n = rr[f"n{i}"] or 0
            TIM.append(dict(pair=lab, period=per_lab, customer_periods=int(n),
                            match_1pct=(rr[f"a{i}"] or 0) / max(n, 1), match_5pct=(rr[f"b{i}"] or 0) / max(n, 1)))
    TIM = pd.DataFrame(TIM)
    TIMP = TIM.pivot_table(index="pair", columns="period", values="match_5pct", sort=False).reset_index()
    TIMP = TIMP[["pair"] + [c for c in ["1 month", "2 months", "quarter"] if c in TIMP.columns]]
DAYS = pd.DataFrame()
if CONFIG["TIMING_DAYS_MONTHS"]:
    with step("share of dollars on the first and last posting days (daily edges)"):
        try:
            _dm = MONTHS[-CONFIG["TIMING_DAYS_MONTHS"]:]
            ed = (spark.table(CONFIG["EDGE_DAILY"]).where(F.col("month").isin(_dm))
                  .select("month", "txn_date", F.regexp_extract(F.col("category").cast("string"), r"^\s*(\d[A-Ca-c]\d?)", 1).alias("code"),
                          F.col("amount").cast("double").alias("amt")))
            _dd = ed.groupBy("month", "txn_date").agg(F.sum("amt").alias("a"))
            _w1 = W.partitionBy("month").orderBy(F.desc("txn_date"))
            _w2 = W.partitionBy("month").orderBy("txn_date")
            _dd = (_dd.withColumn("rl", F.row_number().over(_w1)).withColumn("rf", F.row_number().over(_w2))
                   .select("month", "txn_date", (F.col("rl") <= 2).alias("last2"), (F.col("rf") <= 2).alias("first2"),
                           F.lit(1).alias("one")))
            _nd = _dd.groupBy("month").agg(F.count(F.lit(1)).alias("days")).toPandas()
            DAYS = (ed.join(F.broadcast(_dd), ["month", "txn_date"])
                    .groupBy("code").agg(F.sum("amt").alias("dollars"),
                                         F.sum(F.when(F.col("last2"), F.col("amt")).otherwise(0.0)).alias("l2"),
                                         F.sum(F.when(F.col("first2"), F.col("amt")).otherwise(0.0)).alias("f2"))
                    .toPandas().pipe(_nodec))
            DAYS = DAYS[DAYS.code.str[0].isin(["1", "2", "3", "4", "5", "6", "7"])].copy()
            DAYS["rail"] = DAYS.code.str[0].map(CONFIG["RAIL_OF"])
            DAYS["last_2_posting_days"] = DAYS.l2 / DAYS.dollars
            DAYS["first_2_posting_days"] = DAYS.f2 / DAYS.dollars
            DAYS["uniform_would_be"] = 2 / max(float(_nd.days.median()), 1)
            DAYS = DAYS.sort_values("code")[["rail", "code", "dollars", "first_2_posting_days", "last_2_posting_days", "uniform_would_be"]]
        except Exception as ex:
            DAYS = pd.DataFrame([dict(status=f"daily edges not usable: {type(ex).__name__}: {str(ex)[:120]}")])
TIM.to_csv(os.path.join(OUT_DIR, "timing_matches.csv"), index=False)
_t1 = TIM.pivot_table(index="pair", columns="period", values="match_1pct", sort=False).reset_index()
CELL.show(("Share matching within 5% when months are summed (customers in both sources every month of the period)",
           pct_tbl(TIMP, [c for c in TIMP.columns if c != "pair"])),
          ("Same at 1%", pct_tbl(_t1, [c for c in ["1 month", "2 months", "quarter"] if c in _t1.columns])),
          (f"Share of each category's dollars on the first and last two posting days (last {CONFIG['TIMING_DAYS_MONTHS']} months)",
           pct_tbl(DAYS, ["first_2_posting_days", "last_2_posting_days", "uniform_would_be"]) if "code" in DAYS else DAYS))
for lab in TIMP.pair:
    r = TIMP[TIMP.pair == lab].iloc[0]
    ans(f"Timing, {lab}: match within 5% by 1 month / 2 months / quarter",
        " / ".join(_pf(r.get(c)) for c in ["1 month", "2 months", "quarter"]))

## §11 · Checks paid

Checks paid are the one rail where the gap is broad (half the dollars, a third of the count). Per customer-month
(customers in both sources): is the customer missing from PKG checks entirely, partly, or matched? And by the view's
average check size: if PKG's share of the count is lower for small checks, PKG holds only some kinds of check
(for example image-captured ones).

In [ ]:
CELL.start("§11 · Checks paid")
with step("checks paid per customer-month"):
    _cv = [c for c in RAILS.get("check out", {}).get("core", []) if c in AMT_COLS]
    _cu = [UNIT_OF[c] for c in _cv if c in UNIT_OF]
    if _cv:
        va, vu = view_sum(_cv), (sum((F.col(c) for c in _cu), F.lit(0.0)) if _cu else F.lit(None).cast("double"))
        pa, pn = pkg_sum(["7"], ["ext", "int", "same"], "out"), pkg_sum(["7"], ["ext", "int", "same"], "out", "n")
        C = BOTH.select("pid", "month", va.alias("va"), vu.alias("vu"), pa.alias("pa"), pn.alias("pn")).where((F.col("va") > 0) | (F.col("pa") > 0))
        cls = (F.when(F.col("va") <= 0, "PKG only (view zero)").when(F.col("pa") <= 0, "view only (no PKG checks)")
               .when(F.abs(F.col("pa") - F.col("va")) <= 0.05 * F.greatest("pa", "va"), "match within 5%")
               .when(F.col("pa") < F.col("va"), "PKG lower").otherwise("PKG higher"))
        CLS = (C.withColumn("cls", cls).groupBy("cls").agg(F.count(F.lit(1)).alias("customer_months"), F.sum("va").alias("view_dollars"),
                                                           F.sum("pa").alias("pkg_dollars"), F.sum("vu").alias("view_checks"),
                                                           F.sum("pn").alias("pkg_checks")).toPandas().pipe(_nodec))
        CLS["share_of_customer_months"] = CLS.customer_months / CLS.customer_months.sum()
        CLS["share_of_view_dollars"] = CLS.view_dollars / max(CLS.view_dollars.sum(), 1)
        CLS = CLS.sort_values("view_dollars", ascending=False)
        _tb = [(0, 500, "under 500"), (500, 5_000, "500-5K"), (5_000, 50_000, "5K-50K"), (50_000, np.inf, "50K and over")]
        tk = F.col("va") / F.col("vu")
        e = F.lit(_tb[-1][2])
        for lo, hi, lab in reversed(_tb[:-1]):
            e = F.when(tk < hi, F.lit(lab)).otherwise(e)
        TKT = (C.where((F.col("va") > 0) & (F.col("vu") > 0)).withColumn("ticket", e).groupBy("ticket")
               .agg(F.count(F.lit(1)).alias("customer_months"), F.sum("va").alias("view_dollars"), F.sum("pa").alias("pkg_dollars"),
                    F.sum("vu").alias("view_checks"), F.sum("pn").alias("pkg_checks"),
                    F.expr("percentile_approx(CASE WHEN pn > 0 THEN (pa / pn) / (va / vu) END, 0.5)").alias("median_pkg_ticket_over_view_ticket"))
               .toPandas().pipe(_nodec))
        TKT["dollar_share"] = TKT.pkg_dollars / TKT.view_dollars
        TKT["count_share"] = TKT.pkg_checks / TKT.view_checks
        TKT = TKT.set_index("ticket").reindex([b[2] for b in _tb]).reset_index()
        _c7 = _cat[(_cat.rail == "check")].copy()
    else:
        CLS = TKT = _c7 = pd.DataFrame([dict(status="no check-paid column in the view")])
CLS.to_csv(os.path.join(OUT_DIR, "checks_paid_classes.csv"), index=False)
TKT.to_csv(os.path.join(OUT_DIR, "checks_paid_by_ticket.csv"), index=False)
CELL.show(("Customer-months by how PKG checks paid compare with the view",
           pct_tbl(CLS, ["share_of_customer_months", "share_of_view_dollars"]) if "cls" in CLS else CLS),
          ("By the view's average check size (customer-month)",
           pct_tbl(TKT, ["dollar_share", "count_share", "median_pkg_ticket_over_view_ticket"]) if "ticket" in TKT else TKT),
          ("PKG checks by category, direction and other side (whole window, all TM customers)", _c7))
if "cls" in CLS:
    _x = CLS.set_index("cls")
    ans("Checks paid: customer-months with view checks but no PKG checks (share of customer-months / of view dollars)",
        f"{_x.share_of_customer_months.get('view only (no PKG checks)', 0):.1%} / {_x.share_of_view_dollars.get('view only (no PKG checks)', 0):.1%}")
if "ticket" in TKT:
    ans("Checks paid: PKG count share by view check size (under 500 / 500-5K / 5K-50K / 50K+)",
        " / ".join(_pf(v) for v in TKT.count_share))

## §12 · Coverage per customer and month (table)

Writes `COVERAGE_TABLE`, one row per power id and month, for later use as a feature ("how much of this customer can
PKG see") and as a filter: view core dollars, PKG dollars (everything) and their ratio per rail and direction, the
same for all four rails together in and out, the source flags and the name group from §9. Coverage is empty when
the view has no dollars on that rail. Partitioned by month; rewritten in full on each run.

In [ ]:
CELL.start("§12 · Coverage table")
if CONFIG["WRITE_COVERAGE"]:
    with step("write the coverage table"):
        _sl = lambda k: k.lower().replace(" ", "_")
        _cols = []
        for k, r in RAILS.items():
            v, p = view_sum(r["core"]), pkg_sum(DIG[k], ["ext", "int", "same"], DIR[k])
            _cols += [v.alias(f"view_{_sl(k)}"), p.alias(f"pkg_{_sl(k)}")]
        cov = (J.join(NAME, "pid", "left") if NAME is not None else J.withColumn("customer_name", F.lit(None).cast("string")))
        cov = cov.withColumn("name_group", _grp_expr()).select(
            F.col("pid").alias("customer_pwr_id"), "customer_name", "name_group",
            F.col("in_view").cast("int").alias("in_view"), F.col("in_pkg").cast("int").alias("in_pkg"),
            F.col("n_acct").cast("int").alias("n_acct_view"), *_cols, "month")
        for d in ("in", "out"):
            ks = [k for k in RAILS if DIR[k] == d]
            cov = (cov.withColumn(f"view_all_{d}", sum((F.col(f"view_{_sl(k)}") for k in ks), F.lit(0.0)))
                   .withColumn(f"pkg_all_{d}", sum((F.col(f"pkg_{_sl(k)}") for k in ks), F.lit(0.0))))
        _ratios = [_sl(k) for k in RAILS] + ["all_in", "all_out"]
        for x in _ratios:
            cov = cov.withColumn(f"cov_{x}", F.when(F.col(f"view_{x}") > 0, F.col(f"pkg_{x}") / F.col(f"view_{x}")))
        cov = cov.select(*[c for c in cov.columns if c != "month"], "month")
        spark.sparkContext._jsc.hadoopConfiguration().set("dfs.replication", "2")
        _db = CONFIG["COVERAGE_TABLE"].split(".")[0]
        _loc = next((r[1] for r in spark.sql(f"DESCRIBE DATABASE {_db}").collect() if str(r[0]).lower() in ("location", "database location")), None)
        _w = cov.repartition("month").write.mode("overwrite").partitionBy("month").format("parquet")
        if _loc:
            _w = _w.option("path", f"{_loc.rstrip('/')}/{CONFIG['COVERAGE_TABLE'].split('.')[1]}")
        spark.sql(f"DROP TABLE IF EXISTS {CONFIG['COVERAGE_TABLE']}")
        _w.saveAsTable(CONFIG["COVERAGE_TABLE"])
        ct = spark.table(CONFIG["COVERAGE_TABLE"])
        _q = []
        for x in _ratios:
            q = (ct.where(F.col(f"cov_{x}").isNotNull() & (F.col("in_pkg") == 1))
                 .agg(F.count(F.lit(1)).alias("n"), F.expr(f"percentile_approx(cov_{x}, array(0.1, 0.25, 0.5, 0.75, 0.9))").alias("q"),
                      F.avg(F.when(F.abs(F.col(f"cov_{x}") - 1) <= 0.05, 1).otherwise(0)).alias("near1"),
                      F.avg(F.when(F.col(f"cov_{x}") < 0.5, 1).otherwise(0)).alias("below_half")).first())
            _q.append(dict(coverage=x, customer_months=q["n"], p10=q["q"][0] if q["q"] else None, p25=q["q"][1] if q["q"] else None,
                           median=q["q"][2] if q["q"] else None, p75=q["q"][3] if q["q"] else None, p90=q["q"][4] if q["q"] else None,
                           within_5pct_of_1=q["near1"], below_half=q["below_half"]))
        COVQ = pd.DataFrame(_q)
        _nrow = ct.count()
    CELL.show(("Table written", pd.DataFrame([dict(table=CONFIG["COVERAGE_TABLE"], rows=_nrow, months=len(MONTHS),
                                                   columns=len(ct.columns), impala=f"INVALIDATE METADATA {CONFIG['COVERAGE_TABLE']}")])),
              ("Coverage (PKG ÷ view) across customer-months with view dollars and PKG payments",
               pct_tbl(COVQ, ["p10", "p25", "median", "p75", "p90", "within_5pct_of_1", "below_half"])))
    ans("Coverage table", f"{CONFIG['COVERAGE_TABLE']}: {_nrow:,} rows")
else:
    CELL.show(("Coverage table", pd.DataFrame([dict(status="skipped (WRITE_COVERAGE = False)")])))

## §13 · Answer sheet

In [ ]:
CELL.start("§13 · Answer sheet")
pd.DataFrame(ANSWERS).to_csv(os.path.join(OUT_DIR, "answers.csv"), index=False)
CELL.show(("Answers", pd.DataFrame(ANSWERS)), ("Gates", gates_frame()),
          (f"Files in {OUT_DIR}", pd.DataFrame(FIG_INDEX, columns=["file", "caption"])),
          ("Timings (minutes)", pd.DataFrame(TIMINGS, columns=["step", "minutes"])))